<a href="https://colab.research.google.com/github/phamquocanh149/SMS_SPAM_Classification/blob/main/SMS_Spam_RNN_fixed_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

In [2]:
nltk.download('stopwords')
nltk.download('punkt_tab')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
path = "https://drive.google.com/uc?id=1qKjiO85UhWBaAQmqT8lyyuq09mXLkiym"

# Read the TSV (tab-separated) file
data = pd.read_csv(path, sep=',', engine='python', encoding='ISO-8859-1', header=None, names=['label', 'message', 'col3', 'col4', 'col5'])

print(data.head())

  label                                            message col3 col4 col5
0    v1                                                 v2  NaN  NaN  NaN
1   ham  Go until jurong point, crazy.. Available only ...  NaN  NaN  NaN
2   ham                      Ok lar... Joking wif u oni...  NaN  NaN  NaN
3  spam  Free entry in 2 a wkly comp to win FA Cup fina...  NaN  NaN  NaN
4   ham  U dun say so early hor... U c already then say...  NaN  NaN  NaN


In [4]:
print(f"Số cột dữ liệu: {data.columns.tolist()}")

Số cột dữ liệu: ['label', 'message', 'col3', 'col4', 'col5']


In [5]:
data = data.drop(['col3', 'col4', 'col5'], axis=1)
print(f"Số cột dữ liệu: {data.columns.tolist()}")

Số cột dữ liệu: ['label', 'message']


In [6]:
print(data.shape)
print(data.head())

(5573, 2)
  label                                            message
0    v1                                                 v2
1   ham  Go until jurong point, crazy.. Available only ...
2   ham                      Ok lar... Joking wif u oni...
3  spam  Free entry in 2 a wkly comp to win FA Cup fina...
4   ham  U dun say so early hor... U c already then say...


In [7]:
def preprocessing_text(text):
  text = str(text)
  text = text.lower()
  text = re.sub(r"http\S+|www.\S+", "", text)
  text = re.sub(r"[^\w\s]", '', text)
  text = re.sub(r"\s+", " ",text).strip()
  word = [word for word in text.split() if word not in stop_words]
  return " ".join(word)


In [8]:
data['message'] = data['message'].apply(preprocessing_text)

In [9]:
X = data['message'][1:].tolist()
y = data['label'][1:].tolist()

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

In [12]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14)

In [14]:
leng = [len(s.split()) for s in X_train]
max_len = max(leng)
print(max_len)

72


In [15]:
from collections import Counter
counter = Counter(leng)

for length, count in sorted(counter.items()):
    print(f"{length}: {count} câu")

0: 3 câu
1: 66 câu
2: 232 câu
3: 415 câu
4: 528 câu
5: 443 câu
6: 336 câu
7: 309 câu
8: 220 câu
9: 194 câu
10: 155 câu
11: 149 câu
12: 137 câu
13: 149 câu
14: 140 câu
15: 127 câu
16: 129 câu
17: 118 câu
18: 123 câu
19: 82 câu
20: 105 câu
21: 74 câu
22: 46 câu
23: 36 câu
24: 38 câu
25: 18 câu
26: 20 câu
27: 7 câu
28: 4 câu
29: 6 câu
30: 8 câu
31: 4 câu
32: 2 câu
33: 5 câu
34: 1 câu
35: 2 câu
36: 3 câu
37: 1 câu
38: 4 câu
39: 2 câu
40: 2 câu
42: 2 câu
45: 1 câu
48: 1 câu
49: 2 câu
58: 1 câu
59: 1 câu
61: 2 câu
66: 2 câu
71: 1 câu
72: 1 câu


In [16]:
max_len = 20

In [17]:
token = [word_tokenize(s) for s in X_train]

In [18]:
word_count = Counter(word for sen in token for word in sen)

In [19]:
voca = {word: i+2 for i,word in enumerate(word_count.items())}
voca["<pad>"] = 0
voca["<unk>"] = 1

In [20]:
def encode_sen (text, voca, max_length = max_len):
  text = word_tokenize(text)
  encode = [voca.get(word, voca["<unk>"]) for word in text]
  if len(encode) < max_length:
    encode += [voca['<pad>']] * (max_length - len(encode))
  else:
    encode = encode[:max_length]
  return encode

In [21]:
encode_X_train = [encode_sen(sen, voca) for sen in X_train]
encode_X_test = [encode_sen(sen, voca) for sen in X_test]

In [22]:
smote = SMOTE(random_state=14, sampling_strategy= 0.5)
encode_X_train, y_train = smote.fit_resample(encode_X_train, y_train)

In [23]:
print(y_train)

[0 0 1 ... 1 1 1]


In [24]:
class TextDataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype=torch.long)
    self.y = torch.tensor(y, dtype=torch.long)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

  def __len__(self):
    return len(self.X)

In [25]:
train = TextDataset(encode_X_train, np.array(y_train, dtype=int))
test = TextDataset(encode_X_test, np.array(y_test, dtype=int))

In [26]:
train_loader = DataLoader(train, batch_size=4, shuffle=True)
test_loader = DataLoader(test, batch_size=4, shuffle=False)

In [27]:
class RNNClassifier (nn.Module):
  def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers = 2):
    super(RNNClassifier, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_dim, output_dim)
  def forward(self, x):
    embedded = self.embedding(x)
    output, hidden = self.rnn(embedded)
    # Get the hidden state of the last layer for all batches
    last_hidden = hidden[-1]
    logits = self.fc(last_hidden)
    return logits
  def init_hidden(self, batch_size, device): #khoi tao hidden dau tien
    return torch.zeros(1, batch_size, self.rnn.hidden_size).to(device)

In [28]:
def train_with_logging(model, data_loader, optimizer, criterion, seq_len, epochs, device):
    model.train()
    losses = []
    best_loss = float('inf')   # Khởi tạo loss tốt nhất
    best_model_state = None    # Để lưu lại trạng thái tốt nhất

    for epoch in range(epochs):
        total_loss = 0
        for batch in data_loader:
            inputs, targets = batch
            inputs, targets = inputs.to(device), targets.to(device)

            batch_size = inputs.size(0)

            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)
        losses.append(avg_loss)

        # Kiểm tra và lưu lại best loss
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_model_state = model.state_dict()
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f} (best so far)")
        else:
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return losses, best_loss

In [29]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = RNNClassifier(
    vocab_size=len(voca),
    embedding_dim=256,
    hidden_dim=256,
    output_dim=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00005)


train_with_logging(model, train_loader, optimizer, criterion, seq_len=50, epochs=10, device=device)

Epoch 1, Loss: 0.4441 (best so far)
Epoch 2, Loss: 0.4313 (best so far)
Epoch 3, Loss: 0.4280 (best so far)
Epoch 4, Loss: 0.4266 (best so far)
Epoch 5, Loss: 0.4244 (best so far)
Epoch 6, Loss: 0.4246
Epoch 7, Loss: 0.4208 (best so far)
Epoch 8, Loss: 0.4223
Epoch 9, Loss: 0.4227
Epoch 10, Loss: 0.4219


([0.44409889732219654,
  0.43125346890048527,
  0.42797420898804794,
  0.42658353254859926,
  0.424360257950989,
  0.42457071297149523,
  0.42076102994403036,
  0.4222683752793367,
  0.4227327228015855,
  0.4218652152320162],
 0.42076102994403036)

In [30]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        logits= model(inputs)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


cm = confusion_matrix(all_labels, all_preds)
class_names = label_encoder.classes_.astype(str)


print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_.astype(str)))


Classification Report:
              precision    recall  f1-score   support

         ham       0.97      0.83      0.89       979
        spam       0.39      0.81      0.53       136

    accuracy                           0.82      1115
   macro avg       0.68      0.82      0.71      1115
weighted avg       0.90      0.82      0.85      1115

